## LIBRARY

In [36]:
# Import library
import random

## Defining queen class

In [37]:
class Queen:
  def __init__(self, row, col):
    self.row = row
    self.col = col

## Defining board class

In [38]:
class Board:
  def __init__(self, size):
    self.size = size
    self.queens = [Queen(i, random.randint(0, size - 1)) for i in range(size)]

  def fitness(self):
    # Calculate the number of non-attacking pairs of queens
    # Higher fitness is better (fewer conflicts)
    attacks = 0
    for i in range(self.size):
      for j in range(i + 1, self.size):
          qi = self.queens[i]
          qj = self.queens[j]

          # Check if queens attack each other (same column or diagonal)
          if qi.col == qj.col:  # Same column
              attacks += 1
          elif abs(qi.row - qj.row) == abs(qi.col - qj.col):  # Diagonal
              attacks += 1

    # Maximum possible attacks for n queens is n*(n-1)/2
    max_attacks = (self.size * (self.size - 1)) // 2

    # Fitness is inversely proportional to number of attacks
    # Perfect solution has fitness = max_attacks (no attacks)
    return max_attacks - attacks

  def crossover(self, other):
    # Two-point crossover
    n = self.size
    c1 = random.randint(1, n-2)  # First crossover point
    c2 = random.randint(c1+1, n-1)  # Second crossover point

    child = Board(self.size)

    # Take queens from first parent up to first crossover point
    for i in range(0, c1):
      child.queens[i].col = self.queens[i].col

    # Take queens from second parent between first and second crossover points
    for i in range(c1, c2):
      child.queens[i].col = other.queens[i].col

    # Take queens from first parent after second crossover point
    for i in range(c2, n):
      child.queens[i].col = self.queens[i].col

    return child

  def mutate(self, mutation_rate):
    # Randomly change column positions based on mutation rate
    for i in range(self.size):
      if random.random() < mutation_rate:
          # Move queen to a random column in the same row
          self.queens[i].col = random.randint(0, self.size - 1)
    return self

# Example usage:
board_size = 16
population_size = 256
mutation_rate = 0.2

# Random selection based on fitness-fn
def random_selection(population, fitness_fn):
  # Tournament selection
  tournament_size = 5
  selected = random.sample(population, tournament_size)
  selected.sort(key=fitness_fn, reverse=True)
  return selected[0]

In [39]:
# Reproduce function
def reproduce(x, y):
  return x.crossover(y)

## Implement genetic algorithm

In [40]:
def genetic_algorithm(board_size, population_size, mutation_rate, generations):
  # Initial population
  population = [Board(board_size) for _ in range(population_size)]
  _ = 0
  for generation in range(generations):
    # Get current best solution
    population.sort(key=lambda x: x.fitness(), reverse=True)
    best_board = population[0]
    best_fitness = best_board.fitness()

    # Print progress
    max_fitness = (board_size * (board_size - 1)) // 2

    if _ < best_fitness:
      _ = best_fitness
      print(f"Generation {generation}: Best fitness = {best_fitness}/{max_fitness}")

    # Check if optimal solution found
    if best_fitness == max_fitness:
      print("Found optimal solution!")
      return best_board

    # Create new population
    new_population = []

    for i in range(population_size):
      # Select parents
      x = random_selection(population, lambda board: board.fitness())
      y = random_selection(population, lambda board: board.fitness())

      # Reproduce
      child = reproduce(x, y)

      # Mutate
      if random.random() < mutation_rate:
        child.mutate(mutation_rate)

      new_population.append(child)

    population = new_population

  # Return best solution after all generations
  population.sort(key=lambda x: x.fitness(), reverse=True)
  return population[0]

In [41]:
solution = genetic_algorithm(board_size, population_size, mutation_rate, generations = 500)

Generation 0: Best fitness = 111/120
Generation 2: Best fitness = 115/120
Generation 6: Best fitness = 116/120
Generation 9: Best fitness = 117/120
Generation 15: Best fitness = 118/120
Generation 21: Best fitness = 119/120
Generation 22: Best fitness = 120/120
Found optimal solution!


In [42]:
print("Final solution:")
board = [["." for _ in range(board_size)] for _ in range(board_size)]
for queen in solution.queens:
  board[queen.row][queen.col] = "Q"

for row in board:
  print(" ".join(row))

# Print queen positions
print("\nQueen positions (columns):")
print([queen.col + 1 for queen in solution.queens])

Final solution:
. . . Q . . . . . . . . . . . .
. . . . . . . . . . . . . . Q .
. . . . . . . Q . . . . . . . .
. . . . . . . . . . Q . . . . .
. . . . Q . . . . . . . . . . .
. . . . . . . . . Q . . . . . .
. Q . . . . . . . . . . . . . .
. . . . . . . . . . . . . . . Q
. . Q . . . . . . . . . . . . .
. . . . . . . . Q . . . . . . .
. . . . . . Q . . . . . . . . .
Q . . . . . . . . . . . . . . .
. . . . . . . . . . . . . Q . .
. . . . . . . . . . . Q . . . .
. . . . . Q . . . . . . . . . .
. . . . . . . . . . . . Q . . .

Queen positions (columns):
[4, 15, 8, 11, 5, 10, 2, 16, 3, 9, 7, 1, 14, 12, 6, 13]
